# Worksheet 3.1: Five Mystery Models

Last class you looked inside an AI and watched it choose words by probability. So where does a helpful, polite, useful chatbot come from? Something has to happen between "predicts the next word" and "answers your question".

Today you meet five AI models. All five are built on the same foundational model, which means they all went through the same pre-training on the same pile of text. **Your challeng is to figure out what makes them different.**

**Reminder: Make a copy!** **File > Save a copy in Drive**, close the old tab, then **Share** the copy with everyone in your screen and the instructor as **Editors**. Only one person should have the notebook open at a time.

**Work in screens.** The Driver types, the Navigator reads each instruction and exercise out loud and says what to do, and a third person (if any) is the Checker. Drivers will switch partway through the worksheet.

## Load the five models

**Run the cell below immediately.** It loads all five models into this notebook, which takes a minute or two. Keep reading while the models load.

In [ ]:
#@title Run this cell first (takes a minute or two)

import time
_t_start = time.time()

import torch
import textwrap
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as _hf_logging
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import disable_progress_bars
from safetensors.torch import load_file
_hf_logging.set_verbosity_error()
disable_progress_bars()

_device = "cuda" if torch.cuda.is_available() else "cpu"
_dtype = torch.float16 if _device == "cuda" else torch.float32
if _device == "cpu":
    print("⚠️  No GPU today, so the models will answer very slowly.")
    print("    Try Runtime > Change runtime type > T4 GPU, then run this cell again.")
    print("    If Colab says no GPU is available, tell the teaching team.\n")

def _load(repo):
    model = AutoModelForCausalLM.from_pretrained(repo)
    return model.to(device=_device, dtype=_dtype).eval()

def _tune(parent, adapter_repo):
    """Another model: a copy of one loaded above, plus the changes a fine-tuning made to it."""
    import json, copy
    model = copy.deepcopy(parent).float()
    config = json.load(open(hf_hub_download(adapter_repo, "adapter_config.json")))
    scale = config["lora_alpha"] / config["r"]
    lora = load_file(hf_hub_download(adapter_repo, "adapter_model.safetensors"))
    params = dict(model.named_parameters())
    merged = 0
    with torch.no_grad():
        for key in lora:
            if ".lora_A." in key:
                target = key.replace("base_model.model.", "", 1).replace(".lora_A.weight", ".weight")
                update = lora[key.replace(".lora_A.", ".lora_B.")].float() @ lora[key].float()
                # .to(dtype) keeps the tensor where it is, and the adapter loads on the CPU
                # while the parent is on the GPU, so the device has to be named too.
                params[target].add_((scale * update).to(device=params[target].device,
                                                        dtype=params[target].dtype))
                merged += 1
    if merged != len(lora) // 2:
        raise RuntimeError(f"fine-tuning did not apply: {merged} of {len(lora) // 2} changes")
    return model.to(device=_device, dtype=_dtype).eval()

# The four sets of weights, in no particular order. No peeking: which is which is Part C's puzzle.
print("Loading model 1 of 5...")
_tok1 = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
_m1 = _load("Qwen/Qwen2.5-0.5B")

print("Loading model 2 of 5...")
_tok2 = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
_m2 = _load("Qwen/Qwen2.5-0.5B-Instruct")

print("Loading model 3 of 5...")
_m3 = _tune(_m1, "statisfactions/cheeky-student-base")

print("Loading model 4 of 5...")
_m4 = _tune(_m2, "statisfactions/cheeky-student-instruct")

print("Setting up model 5 of 5...")

def _wrap(text, width=80):
    lines = []
    for line in text.split("\n"):
        lines.extend(textwrap.wrap(line, width=width) if len(line) > width else [line])
    return "\n".join(lines)

def _generate(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt").to(_device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=True,
                             temperature=0.5, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def _plain(model, tokenizer, prompt):
    """No template at all: the prompt is just text to continue."""
    return prompt + _generate(model, tokenizer, prompt)

def _labels(model, tokenizer, prompt):
    """The prompt is wrapped in User: and Assistant: labels."""
    return _generate(model, tokenizer, f"User: {prompt}\nAssistant:").split("User:")[0].strip()

def _chat(model, tokenizer, prompt):
    """The prompt is wrapped in the chat format the model was trained on."""
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
    return _generate(model, tokenizer, text).strip()

# No peeking. Which model is which is Part C's puzzle.
_MODELS = {
    "A": lambda p: _chat(_m2, _tok2, p),
    "B": lambda p: _chat(_m4, _tok2, p),
    "C": lambda p: _labels(_m1, _tok1, p),
    "D": lambda p: _labels(_m3, _tok1, p),
    "E": lambda p: _plain(_m1, _tok1, p),
}

def ask_A(prompt): print(_wrap(_MODELS["A"](prompt)))
def ask_B(prompt): print(_wrap(_MODELS["B"](prompt)))
def ask_C(prompt): print(_wrap(_MODELS["C"](prompt)))
def ask_D(prompt): print(_wrap(_MODELS["D"](prompt)))
def ask_E(prompt): print(_wrap(_MODELS["E"](prompt)))

def compare_all(prompt):
    """Ask all five models the same thing."""
    print("=" * 70)
    print(f"PROMPT: {prompt}")
    for letter in "ABCDE":
        print("=" * 70)
        print(f"MODEL {letter}:")
        print(_wrap(_MODELS[letter](prompt)))
    print("=" * 70)

print(f"\n✓ All five models are ready. That took {time.time() - _t_start:.0f} seconds.")
print("  Use ask_A(...) through ask_E(...), or compare_all(...) for all five at once.")

## If the notebook stops working

Colab gives free notebooks a limited amount of time, and it sometimes takes the models away while you are still working. The usual sign is an error like `NameError: name 'ask_A' is not defined`, or a message saying you were disconnected.

**The fix is almost always the same: run the Load the five models cell again**, and wait for the ✓ message. Everything you have typed is safe. Your copy lives in your Drive, and it is only the models that went away.

Two things help you avoid it:

- Keep this tab open and in front of you. Colab hangs up on notebooks it thinks nobody is using.
- Do not open the notebook on two devices at once.

**NOTE:** If it keeps happening, or Colab refuses to give you a GPU, raise your hand. Two screens can share one laptop for a while.

# Using AI on this worksheet

We go back and forth in this course between using generative AI and not using it, to balance giving you support with building your own ability. Today the AI is the thing we are studying, and you will be running five of them right here in this notebook, so please don't use any other AI tools like ChatGPT on this worksheet. When a question asks what you noticed or what you think, the answer should come from your own screen.

If you hit an issue, ask your screen and then the teaching team. Hitting issues is part of the learning process, and this worksheet is graded on completeness and effort, so there is nothing to gain from asking a chatbot anyway.

**Exercise 0. Double-click to edit the text cell below, and write "We agree" at the end:**

We, the people in this screen, understand that we will only use AI where this worksheet asks us to, and that our answers about what we noticed and what we think will be our own. ______________

# Part A: Meet five mystery models

The five models are labeled A, B, C, D and E. Your job is to figure out what makes them different. Make sure to **come up with your own prompts** so we can try the models on many different things. Second, **use the same prompts across all five models**, or you have no way of knowing whether the came from the model or from the question.

So write your prompts once and reuse them by name. You did this last class with `my_factual_prompt`.

**Exercise A.1:** Write three prompts of **different kinds**. Varying the kind of task is how differences show up: a factual question, a math problem, a creative task, life advice, an instruction to follow. Decide on them together as a screen.

In [ ]:
# Write your three prompts between the quotation marks.
prompt1 = "___"
prompt2 = "___"
prompt3 = "___"

Now try Model A. Because your prompts have names, you don't have to type them again:

In [ ]:
ask_A(prompt1)

In [ ]:
ask_A(prompt2)

In [ ]:
ask_A(prompt3)

**Exercise A.2: Model A.** Double-click the cell below and respond to each bullet; a few words each is plenty.

- Model A in three words:
- Was it helpful?
- The strangest thing it did:

Now the same three prompts for the other four models. Run a cell again whenever you want another go, since these models pick words by probability, exactly as you saw last class, and the same prompt gives a different answer each time.

### Model B

In [ ]:
ask_B(prompt1)

In [ ]:
ask_B(prompt2)

In [ ]:
ask_B(prompt3)

**Exercise A.3: Model B.** Double-click the cell below and respond to each bullet; a few words each is plenty.

- Model B in three words:
- Was it helpful?
- The strangest thing it did:

### Model C

In [ ]:
ask_C(prompt1)

In [ ]:
ask_C(prompt2)

In [ ]:
ask_C(prompt3)

**Exercise A.4: Model C.** Double-click the cell below and respond to each bullet; a few words each is plenty.

- Model C in three words:
- Was it helpful?
- The strangest thing it did:

### Model D

In [ ]:
ask_D(prompt1)

In [ ]:
ask_D(prompt2)

In [ ]:
ask_D(prompt3)

**Exercise A.5: Model D.** Double-click the cell below and respond to each bullet; a few words each is plenty.

- Model D in three words:
- Was it helpful?
- The strangest thing it did:

### Model E

In [ ]:
ask_E(prompt1)

In [ ]:
ask_E(prompt2)

In [ ]:
ask_E(prompt3)

**Exercise A.6: Model E.** Double-click the cell below and respond to each bullet; a few words each is plenty.

- Model E in three words:
- Was it helpful?
- The strangest thing it did:

## 🔄 Time to swap Drivers

The second Driver takes over now.

- If the new Driver wants to use their own laptop, they open the notebook link, and the old Driver **closes their tab**.
- Or, if you're both happy to, just pass the laptop over.

Either way, only one person types at a time.

**NOTE:** If the new Driver opens the notebook on their own laptop, two cells have to be run again on that laptop: the **Load the five models** cell, and the cell with your three prompts.

# Part B: All five, side by side

Comparing five answers you ran ten minutes apart is hard. `compare_all` sends the same prompt to all five models and prints the answers one after another, so the differences are right next to each other.

**Exercise B.1:** Create a new prompt you think will separate the models most sharply, and use it with `compare_all` in the cell below.

In [ ]:
compare_all("___your prompt here___")

**Exercise B.2:** What additional differences do you notice between the 5 models? Explain.

*Your answer here.*

# Part C: What was done to each one

Here are the five models you have been talking to. All five began as the same foundational model. What differs is what was done to it afterward, and that is what you have been listening for.

| # | What it is | What was done to it |
|---|---|---|
| 1 | **Raw base model** | Nothing at all after pre-training. It only predicts the next word. It has no idea it is supposed to be answering a question, because nobody ever told it there is such a thing as a conversation. |
| 2 | **Base with labels** | The very same raw model, except that we wrap your prompt in "User:" and "Assistant:" before handing it over. No training, just a hint about the format. |
| 3 | **Instruct model** | The base model, trained further on thousands of examples of helpful question-and-answer conversations, by the company that built it. |
| 4 | **Cheeky base** | The raw base model, trained further on 150 funny and absurd answers that students wrote on tests. |
| 5 | **Cheeky instruct** | The helpful instruct model, trained further on those same 150 student answers. |

**Exercise C.1:** Match each description to the letter you met above. Write down the evidence that convinced you, and mark how sure you are. Run `compare_all` again if it helps you decide.

**1. Raw base model**
- Letter:
- Why you think so:
- Sure, or guessing?

**2. Base with labels**
- Letter:
- Why you think so:
- Sure, or guessing?

**3. Instruct model**
- Letter:
- Why you think so:
- Sure, or guessing?

**4. Cheeky base**
- Letter:
- Why you think so:
- Sure, or guessing?

**5. Cheeky instruct**
- Letter:
- Why you think so:
- Sure, or guessing?

🛑 **Stop!** 🛑

We will give you a marker. Write your screen's five letters in your column on the board for us to discuss.  If you have extra time, work on the Shoggoth Field Journal.

# Important: Submission Instructions

1. Check to make sure you've completed all the exercises.
2. Check that the notebook is shared as an **Editor** with everyone in your screen and the instructor.
3. **Each of you** submits on Canvas: paste the notebook link into the **Website URL** box, and in the **Comments** box write whether you drove today (**Drove**, **Didn't drive**, or **N/A**). Everyone in your screen submits the same link.

**Acknowledgments:** This notebook contains materials created by Ethan C. Brown with assistance from Claude (see [conversation 1](https://claude.ai/share/58d28d97-ab83-48e0-b245-d533dc96d0d4), [conversation 2](https://claude.ai/share/108e259e-bb41-47d7-86ff-45b209f40fb1), and [conversation 3](https://claude.ai/share/670cb650-f7d7-45bc-afa9-a89916f13979)).

Current version created by Ethan C. Brown in collaboration with Claude Code.